In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder # what is meesage place holder ??
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory #in built memory in langchain --
from langchain_core.runnables.history import RunnableWithMessageHistory # wait for 15 mint
from langchain_core.messages import HumanMessage, AIMessage # by default your LLM try --> input --> human or AI meeage

In [3]:
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")

In [4]:
history = InMemoryChatMessageHistory()

In [5]:
history.messages

[]

In [8]:
history.add_user_message("I have a billing issue")
history.add_ai_message("Undrstood you query but I can thelp you")

history.add_user_message("I have a billing issue")
history.add_ai_message("Undrstood you query but I can thelp you")

In [9]:
history.messages

[HumanMessage(content='I have a billing issue', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I have a billing issue', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Undrstood you query but I can thelp you', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I have a billing issue', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Undrstood you query but I can thelp you', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [10]:
# MessagesPlaceholder
# a placeholder inside your prompt ( chatprompt templates)

prompt = ChatPromptTemplate.from_messages([
    ("system","You are a professional email support agent"),
    MessagesPlaceholder(variable_name="history"), # slot for past message
    ("human","{input}") # current user message

    ])

prompt

ChatPromptTemplate(input_variables=['history', 'input'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchain_c

In [11]:
past_messages = [
    HumanMessage(content="Billing complaint from John — invoice #1042, overcharged $250."),
    AIMessage(content="Classified: Billing — High Priority. SLA: 2 hours."),
]

formatted = prompt.format_messages(
    history = past_messages,
    input = "Draft an aplology email for John"
)


In [12]:
formatted

[SystemMessage(content='You are a professional email support agent', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Billing complaint from John — invoice #1042, overcharged $250.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Classified: Billing — High Priority. SLA: 2 hours.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Draft an aplology email for John', additional_kwargs={}, response_metadata={})]

In [13]:
# step 1 : defined your propt
prompt = ChatPromptTemplate.from_messages([
    ("system", """\
You are a professional email support agent.
Classify complaints as: Billing / Technical / General.
Priority: High (financial impact > $100 or urgent) / Medium / Low.
Remember all customer details throughout the conversation."""),
    MessagesPlaceholder(variable_name="history"),  # ← past turns injected here
    ("human", "{input}"),                           # ← new message here
])

# step 2 LCEL
chain = prompt | llm | StrOutputParser()

In [ ]:
# step 3 
store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [15]:
# step 4:
email_assistant = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

/Users/rahultiwari/Documents/02_Freelancing/Rbyte_project/Sathish_AI/test-env/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3747: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [16]:
# rahul will login to the system
cfg = {"configurable":{"session_id":"123rahul"}}

r1 = email_assistant.invoke({"input": "I received a billing complaint from John about invoice #1042. He was charged $500 instead of $250."},
                            config=cfg
)

In [21]:
r2 = email_assistant.invoke(
    {"input": "What category does his complaint fall under?"},
    config=cfg
)

In [23]:
r2 = email_assistant.invoke(
    {"input": "how I can resolve this?"},
    config=cfg
)

In [24]:
store['123rahul'].messages

[HumanMessage(content='I received a billing complaint from John about invoice #1042. He was charged $500 instead of $250.', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Complaint Classification: Billing  \nPriority: High  \n\nCustomer Details:  \nName: John  \nInvoice Number: #1042  \nCharge Discrepancy: $500 charged instead of $250  \n\nPlease let me know if you need further assistance with this request or if you'd like to add any additional details.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What category does his complaint fall under?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="John's complaint falls under the **Billing** category.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='how I can resolve this?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="To resolve John's billing complaint regarding invo

In [25]:
cfg = {"configurable":{"session_id":"123sathish"}}

r1 = email_assistant.invoke({"input": "I received a billing complaint from John about invoice #1042. He was charged $500 instead of $250."},
                            config=cfg
)

In [26]:
store

{'123rahul': InMemoryChatMessageHistory(messages=[HumanMessage(content='I received a billing complaint from John about invoice #1042. He was charged $500 instead of $250.', additional_kwargs={}, response_metadata={}), AIMessage(content="Complaint Classification: Billing  \nPriority: High  \n\nCustomer Details:  \nName: John  \nInvoice Number: #1042  \nCharge Discrepancy: $500 charged instead of $250  \n\nPlease let me know if you need further assistance with this request or if you'd like to add any additional details.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What category does his complaint fall under?', additional_kwargs={}, response_metadata={}), AIMessage(content="John's complaint falls under the **Billing** category.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='how I can resolve this?', additional_kwargs={}, response_metadata={}), AIMessage(content="To res